# Práctica 1 — Dataset 2: Renta mediana por Census Tract (Bronze ➜ Silver)

**Objetivo:** Leer el CSV de ingresos desde bronze, filtrar únicamente los Census Tracts de San Francisco,
limpiar/transformar y persistir en silver como Parquet.

In [1]:
# ============================================================
# 0) Imports + SparkSession
# ============================================================
from pyspark.sql import SparkSession, functions as F, types as T
import os

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /opt/spark/ivy/cache
The jars for the packages stored in: /opt/spark/ivy/jars
org.apache.spark#spark-hadoop-cloud_2.13 added as a dependency
io.delta#delta-spark_2.13 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ae9df9bb-1029-40a1-bc10-ad4bcbf91833;1.0
	confs: [default]
	found org.apache.spark#spark-hadoop-cloud_2.13;4.0.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
	found s

Spark version: 4.0.1


In [2]:
# Funcion para facilitar visualización
from IPython.display import display
import pandas as pd
pd.set_option("display.max_columns", None)  # muestra todas las columnas

def display_df(
    df,
    n=10,
    columns=None,
    sort_by=None,
    ascending=False
):
    """
    Visualiza un DataFrame de Spark en Jupyter de forma similar a Databricks display().

    Parámetros:
    - df: DataFrame de Spark
    - n: número máximo de filas a mostrar (default: 20)
    - columns: lista opcional de columnas a mostrar
    - sort_by: columna opcional para ordenar
    - ascending: orden ascendente o descendente
    """

    if columns:
        df = df.select(columns)

    if sort_by:
        df = df.orderBy(sort_by, ascending=ascending)

    pdf = df.limit(n).toPandas()
    display(pdf)


In [3]:
# ============================================================
# 0.1) Configuración MinIO (S3A)
# ============================================================
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minioadmin")
MINIO_USE_SSL = os.getenv("MINIO_USE_SSL", "false").lower() == "true"

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", MINIO_ENDPOINT)
hconf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hconf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "true" if MINIO_USE_SSL else "false")
hconf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

print("MINIO_ENDPOINT =", MINIO_ENDPOINT)

MINIO_ENDPOINT = http://minio:9000


## 1) Lectura desde Bronze (sin inferir esquema)

Se lee **a nivel de carpeta**.

In [4]:
BRONZE_PATH = "s3a://bronze/ACSST5Y2023.S1901-Data.csv/"

df_raw = (
    spark.read
        .option("header", True)
        .option("inferSchema", "false")
        .csv(BRONZE_PATH)
)

df_raw.printSchema()
display_df(df_raw, n=5)


root
 |-- GEO_ID: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- S1901_C01_001E: string (nullable = true)
 |-- S1901_C01_001M: string (nullable = true)
 |-- S1901_C01_002E: string (nullable = true)
 |-- S1901_C01_002M: string (nullable = true)
 |-- S1901_C01_003E: string (nullable = true)
 |-- S1901_C01_003M: string (nullable = true)
 |-- S1901_C01_004E: string (nullable = true)
 |-- S1901_C01_004M: string (nullable = true)
 |-- S1901_C01_005E: string (nullable = true)
 |-- S1901_C01_005M: string (nullable = true)
 |-- S1901_C01_006E: string (nullable = true)
 |-- S1901_C01_006M: string (nullable = true)
 |-- S1901_C01_007E: string (nullable = true)
 |-- S1901_C01_007M: string (nullable = true)
 |-- S1901_C01_008E: string (nullable = true)
 |-- S1901_C01_008M: string (nullable = true)
 |-- S1901_C01_009E: string (nullable = true)
 |-- S1901_C01_009M: string (nullable = true)
 |-- S1901_C01_010E: string (nullable = true)
 |-- S1901_C01_010M: string (nullable = true)
 

26/03/03 11:26:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/03 11:26:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: GEO_ID, NAME, S1901_C01_001E, S1901_C01_001M, S1901_C01_002E, S1901_C01_002M, S1901_C01_003E, S1901_C01_003M, S1901_C01_004E, S1901_C01_004M, S1901_C01_005E, S1901_C01_005M, S1901_C01_006E, S1901_C01_006M, S1901_C01_007E, S1901_C01_007M, S1901_C01_008E, S1901_C01_008M, S1901_C01_009E, S1901_C01_009M, S1901_C01_010E, S1901_C01_010M, S1901_C01_011E, S1901_C01_011M, S1901_C01_012E, S1901_C01_012M, S1901_C01_013E, S1901_C01_013M, S1901_C01_014E, S1901_C01_014M, S1901_C01_015E, S1901_C01_015M, S1901_C01_016E, S1901_C01_016M, S1901_C02_001E, S1901_C02_001M, S1901_C02_002E, S1901_C02_002M, S1901_C02_003E, S1901_C02_003M, S1901_C02_004E, S1901_C02_004M, S1901_C02_005E, S1901_C02_005M, S1901_C02_006E, S1901_C02_006M, 

,GEO_ID,NAME,S1901_C01_001E,S1901_C01_001M,S1901_C01_002E,S1901_C01_002M,S1901_C01_003E,S1901_C01_003M,S1901_C01_004E,S1901_C01_004M,S1901_C01_005E,S1901_C01_005M,S1901_C01_006E,S1901_C01_006M,S1901_C01_007E,S1901_C01_007M,S1901_C01_008E,S1901_C01_008M,S1901_C01_009E,S1901_C01_009M,S1901_C01_010E,S1901_C01_010M,S1901_C01_011E,S1901_C01_011M,S1901_C01_012E,S1901_C01_012M,S1901_C01_013E,S1901_C01_013M,S1901_C01_014E,S1901_C01_014M,S1901_C01_015E,S1901_C01_015M,S1901_C01_016E,S1901_C01_016M,S1901_C02_001E,S1901_C02_001M,S1901_C02_002E,S1901_C02_002M,S1901_C02_003E,S1901_C02_003M,S1901_C02_004E,S1901_C02_004M,S1901_C02_005E,S1901_C02_005M,S1901_C02_006E,S1901_C02_006M,S1901_C02_007E,S1901_C02_007M,S1901_C02_008E,S1901_C02_008M,S1901_C02_009E,S1901_C02_009M,S1901_C02_010E,S1901_C02_010M,S1901_C02_011E,S1901_C02_011M,S1901_C02_012E,S1901_C02_012M,S1901_C02_013E,S1901_C02_013M,S1901_C02_014E,S1901_C02_014M,S1901_C02_015E,S1901_C02_015M,S1901_C02_016E,S1901_C02_016M,S1901_C03_001E,S1901_C03_001M,S1901_C03_002E,S1901_C03_002M,S1901_C03_003E,S1901_C03_003M,S1901_C03_004E,S1901_C03_004M,S1901_C03_005E,S1901_C03_005M,S1901_C03_006E,S1901_C03_006M,S1901_C03_007E,S1901_C03_007M,S1901_C03_008E,S1901_C03_008M,S1901_C03_009E,S1901_C03_009M,S1901_C03_010E,S1901_C03_010M,S1901_C03_011E,S1901_C03_011M,S1901_C03_012E,S1901_C03_012M,S1901_C03_013E,S1901_C03_013M,S1901_C03_014E,S1901_C03_014M,S1901_C03_015E,S1901_C03_015M,S1901_C03_016E,S1901_C03_016M,S1901_C04_001E,S1901_C04_001M,S1901_C04_002E,S1901_C04_002M,S1901_C04_003E,S1901_C04_003M,S1901_C04_004E,S1901_C04_004M,S1901_C04_005E,S1901_C04_005M,S1901_C04_006E,S1901_C04_006M,S1901_C04_007E,S1901_C04_007M,S1901_C04_008E,S1901_C04_008M,S1901_C04_009E,S1901_C04_009M,S1901_C04_010E,S1901_C04_010M,S1901_C04_011E,S1901_C04_011M,S1901_C04_012E,S1901_C04_012M,S1901_C04_013E,S1901_C04_013M,S1901_C04_014E,S1901_C04_014M,S1901_C04_015E,S1901_C04_015M,S1901_C04_016E,S1901_C04_016M,_c130
0,Geography,Geographic Area Name,Estimate!!Households!!Total,Margin of Error!!Households!!Total,"Estimate!!Households!!Total!!Less than $10,000",Margin of Error!!Households!!Total!!Less than ...,"Estimate!!Households!!Total!!$10,000 to $14,999","Margin of Error!!Households!!Total!!$10,000 to...","Estimate!!Households!!Total!!$15,000 to $24,999","Margin of Error!!Households!!Total!!$15,000 to...","Estimate!!Households!!Total!!$25,000 to $34,999","Margin of Error!!Households!!Total!!$25,000 to...","Estimate!!Households!!Total!!$35,000 to $49,999","Margin of Error!!Households!!Total!!$35,000 to...","Estimate!!Households!!Total!!$50,000 to $74,999","Margin of Error!!Households!!Total!!$50,000 to...","Estimate!!Households!!Total!!$75,000 to $99,999","Margin of Error!!Households!!Total!!$75,000 to...","Estimate!!Households!!Total!!$100,000 to $149,999","Margin of Error!!Households!!Total!!$100,000 t...","Estimate!!Households!!Total!!$150,000 to $199,999","Margin of Error!!Households!!Total!!$150,000 t...","Estimate!!Households!!Total!!$200,000 or more","Margin of Error!!Households!!Total!!$200,000 o...",Estimate!!Households!!Median income (dollars),Margin of Error!!Households!!Median income (do...,Estimate!!Households!!Mean income (dollars),Margin of Error!!Households!!Mean income (doll...,Estimate!!Households!!PERCENT ALLOCATED!!House...,Margin of Error!!Households!!PERCENT ALLOCATED...,Estimate!!Households!!PERCENT ALLOCATED!!Famil...,Margin of Error!!Households!!PERCENT ALLOCATED...,Estimate!!Households!!PERCENT ALLOCATED!!Nonfa...,Margin of Error!!Households!!PERCENT ALLOCATED...,Estimate!!Families!!Total,Margin of Error!!Families!!Total,"Estimate!!Families!!Total!!Less than $10,000",Margin of Error!!Families!!Total!!Less than $1...,"Estimate!!Families!!Total!!$10,000 to $14,999","Margin of Error!!Families!!Total!!$10,000 to $...","Estimate!!Families!!Total!!$15,000 to $24,999","Margin of Error!!Families!!Total!!$15,000 to $...","Estimate!!Families!!Total!!$25,000 to $34,999","Margin of Error!!Families!!Total!!$25,000 to $...",

## 2) Filtrar filas de interés (solo Census Tracts de San Francisco)

Nos quedamos con las filas cuyo `GEO_ID` empiece por:

- `1400000US`

(es decir, el nivel de Census Tract).

In [5]:
df_filtered = df_raw.filter(F.col("GEO_ID").startswith("1400000US"))

print("Filas tras filtrado:", df_filtered.count())
display_df(df_filtered.select("GEO_ID", "NAME"), n=5)


Filas tras filtrado: 244


,GEO_ID,NAME
0,1400000US06075010101,Census Tract 101.01; San Francisco County; Cal...
1,1400000US06075010102,Census Tract 101.02; San Francisco County; Cal...
2,1400000US06075010201,Census Tract 102.01; San Francisco County; Cal...
3,1400000US06075010202,Census Tract 102.02; San Francisco County; Cal...
4,1400000US06075010300,Census Tract 103; San Francisco County; Califo...


## 3) Crear columna `census_tract` (string) extrayendo la parte posterior a `US`

Ejemplo: `1400000US06075010101` → `06075010101`

In [6]:
df_with_tract = df_filtered.withColumn(
    "census_tract",
    F.regexp_extract(F.col("GEO_ID"), r"US(\d+)$", 1)
)

display_df(df_with_tract.select("GEO_ID", "census_tract"), n=10)


,GEO_ID,census_tract
0,1400000US06075010101,06075010101
1,1400000US06075010102,06075010102
2,1400000US06075010201,06075010201
3,1400000US06075010202,06075010202
4,1400000US06075010300,06075010300
5,1400000US06075010401,06075010401
6,1400000US06075010402,06075010402
7,1400000US06075010500,06075010500
8,1400000US06075010600,06075010600
9,1400000US06075010701,06075010701


## 4) Seleccionar columnas y renombrar

Columnas requeridas:
- `census_tract`
- `S1901_C01_012E` → `median_household_income_raw`
- `NAME` → `tract_name`

In [7]:
df_sel = df_with_tract.select(
    F.col("census_tract"),
    F.col("S1901_C01_012E").alias("median_household_income_raw"),
    F.col("NAME").alias("tract_name")
)

df_sel.printSchema()
display_df(df_sel, n=10)


root
 |-- census_tract: string (nullable = true)
 |-- median_household_income_raw: string (nullable = true)
 |-- tract_name: string (nullable = true)



,census_tract,median_household_income_raw,tract_name
0,06075010101,101974,Census Tract 101.01; San Francisco County; Cal...
1,06075010102,108417,Census Tract 101.02; San Francisco County; Cal...
2,06075010201,160221,Census Tract 102.01; San Francisco County; Cal...
3,06075010202,206484,Census Tract 102.02; San Francisco County; Cal...
4,06075010300,158973,Census Tract 103; San Francisco County; Califo...
5,06075010401,207500,Census Tract 104.01; San Francisco County; Cal...
6,06075010402,158500,Census Tract 104.02; San Francisco County; Cal...
7,06075010500,171667,Census Tract 105; San Francisco County; Califo...
8,06075010600,86111,Census Tract 106; San Francisco County; Califo...
9,06075010701,27531,Census Tract 107.01; San Francisco County; Cal...


## 5) Crear `median_household_income` (double) a partir de `median_household_income_raw`

- Hay valores especiales como `250,000+` que deben convertirse a `250000`.
- Además, eliminamos separadores `,` y cualquier carácter no numérico.

In [8]:
# Nos quedamos con los dígitos (esto convierte "250,000+" -> "250000")
income_digits = F.regexp_replace(F.col("median_household_income_raw"), r"[^0-9]", "")

df_typed = df_sel.withColumn(
    "median_household_income",
    F.when(income_digits == "", F.lit(None).cast("double"))
     .otherwise(income_digits.cast("double"))
)

display_df(df_typed.select("median_household_income_raw", "median_household_income"), n=20)


,median_household_income_raw,median_household_income
0,101974,101974.0
1,108417,108417.0
2,160221,160221.0
3,206484,206484.0
4,158973,158973.0
5,207500,207500.0
6,158500,158500.0
7,171667,171667.0
8,86111,86111.0
9,27531,27531.0


## 6) Crear `income_is_censored` (boolean)

- `true` cuando `median_household_income_raw` sea exactamente `250,000+`
- `false` en cualquier otro caso

In [9]:
df_final = df_typed.withColumn(
    "income_is_censored",
    (F.col("median_household_income_raw") == F.lit("250,000+")).cast("boolean")
)

df_final.printSchema()
display_df(df_final, n=10)


root
 |-- census_tract: string (nullable = true)
 |-- median_household_income_raw: string (nullable = true)
 |-- tract_name: string (nullable = true)
 |-- median_household_income: double (nullable = true)
 |-- income_is_censored: boolean (nullable = true)



,census_tract,median_household_income_raw,tract_name,median_household_income,income_is_censored
0,06075010101,101974,Census Tract 101.01; San Francisco County; Cal...,101974.0,False
1,06075010102,108417,Census Tract 101.02; San Francisco County; Cal...,108417.0,False
2,06075010201,160221,Census Tract 102.01; San Francisco County; Cal...,160221.0,False
3,06075010202,206484,Census Tract 102.02; San Francisco County; Cal...,206484.0,False
4,06075010300,158973,Census Tract 103; San Francisco County; Califo...,158973.0,False
5,06075010401,207500,Census Tract 104.01; San Francisco County; Cal...,207500.0,False
6,06075010402,158500,Census Tract 104.02; San Francisco County; Cal...,158500.0,False
7,06075010500,171667,Census Tract 105; San Francisco County; Califo...,171667.0,False
8,06075010600,86111,Census Tract 106; San Francisco County; Califo...,86111.0,False
9,06075010701,27531,Census Tract 107.01; San Francisco County; Cal...,27531.0,False


## 7) Escritura en Silver (Parquet)

Persistimos el DataFrame final en:

- `silver/sf_median_household_income/`

In [10]:
SILVER_PATH = "s3a://silver/sf_median_household_income/"

(
    df_final.write
      .mode("overwrite")
      .parquet(SILVER_PATH)
)


df_check = spark.read.parquet(SILVER_PATH)
print("Filas en silver:", df_check.count())
df_check.printSchema()

26/03/03 11:26:49 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to sf_median_household_income/_temporary/0/_temporary/attempt_202603031126488509670695705494964_0010_m_000000_9/part-00000-124cc8ea-60ee-4f5e-bfff-5b3627b90495-c000.snappy.parquet. This is Unsupported
                                                                                

Filas en silver: 244
root
 |-- census_tract: string (nullable = true)
 |-- median_household_income_raw: string (nullable = true)
 |-- tract_name: string (nullable = true)
 |-- median_household_income: double (nullable = true)
 |-- income_is_censored: boolean (nullable = true)



## 8) Preguntas analíticas

### i) ¿Cuántos census tracts hay en total tras el filtrado?

In [11]:
total_tracts = df_final.select("census_tract").distinct().count()

print(total_tracts)

244


Hay 244 census tracts distintos tras el filtrado (es decir GEO_ID's que empieza por 1400000US y no se repiten).


### ii) ¿Cuántos census tracts no tienen información de renta (`median_household_income` nulo)?

In [12]:
tracts_income_null = df_final.filter(F.col("median_household_income").isNull()).count()
print(tracts_income_null)

7


El número de tracts con median_household_income nulo son 7.


### iii) ¿Cuántos tracts tienen la renta censurada (`income_is_censored = true`)?

In [13]:
tracts_censored = df_final.filter(F.col("income_is_censored") == True).count()
print(tracts_censored)

6


El número de tracts con income_is_censored = true es de 6. Esos casos representan 250,000+ (valor real ≥ 250000).

### iv) ¿Cuál es el mínimo, máximo y media de `median_household_income` (ignorando nulos)?

In [18]:
stats = (
    df_final
    .select(
        F.min("median_household_income").alias("min_income"),
        F.max("median_household_income").alias("max_income"),
        F.avg("median_household_income").alias("avg_income"),
    )
    .collect()[0]
    
)
print(stats)

Row(min_income=13569.0, max_income=250000.0, avg_income=144302.4345991561)


Los valores mínimo, máximo y media (ignorando nulos automáticamente) son 13569, 250000 y 144302 respectivamente. Al convertir 250,000+ a 250000, la media puede infraestimar la renta real en esos tracts censurados. Podemos ver que la media esta bastante mas cerca del minimo que del máximo, esto nos proporciona una información de análisis muy util sobre las rentas de San Francisco
